In [1]:
from pathlib import Path
from PIL import Image, ImageOps
from collections import Counter
import xml.etree.ElementTree as ET
import csv
import math

classes = ["D00", "D10", "D20", "D40"]
context = 0.15
size = 224
threshold = 0.30
countries = ["Japan", "Czech"]

for country in countries:
    for group in ["clean", "ambiguous"]:
        for label in classes:
            Path(f"data/train/{country}/processed_raw/{group}/{label}").mkdir(parents=True, exist_ok=True)

    counts = Counter()
    rows = []

    # Doc anh va nhan
    for xml_path in sorted(Path(f"data/train/{country}/annotations/xmls").glob("*.xml")):
        root = ET.parse(xml_path).getroot()
        filename = root.findtext("filename")
        image_path = Path(f"data/train/{country}/images") / filename

        with Image.open(image_path) as img:
            image = img.convert("RGB")
        width, height = image.size
        objects = []

        for obj in root.findall("object"):
            label = obj.findtext("name").strip()
            box = obj.find("bndbox")
            xmin = float(box.findtext("xmin"))
            ymin = float(box.findtext("ymin"))
            xmax = float(box.findtext("xmax"))
            ymax = float(box.findtext("ymax"))

            xmin = max(0, min(xmin, width))
            ymin = max(0, min(ymin, height))
            xmax = max(0, min(xmax, width))
            ymax = max(0, min(ymax, height))
            if xmax <= xmin or ymax <= ymin:
                continue
            objects.append((label, xmin, ymin, xmax, ymax))

        for index, (label, xmin, ymin, xmax, ymax) in enumerate(objects):
            if label not in classes:
                continue

            # Cat vung hu hong va them boi canh
            box_width = xmax - xmin
            box_height = ymax - ymin
            side = math.ceil(max(box_width, box_height) * (1 + 2 * context))
            if side <= width and side <= height:
                left = round((xmin + xmax) / 2 - side / 2)
                top = round((ymin + ymax) / 2 - side / 2)
                left = max(0, min(left, width - side))
                top = max(0, min(top, height - side))
                right, bottom = left + side, top + side
            else:
                left = max(0, math.floor(xmin - box_width * context))
                top = max(0, math.floor(ymin - box_height * context))
                right = min(width, math.ceil(xmax + box_width * context))
                bottom = min(height, math.ceil(ymax + box_height * context))

            # Kiem tra crop co lan nhan khac khong
            ambiguous = False
            for other_label, other_left, other_top, other_right, other_bottom in objects:
                if other_label == label:
                    continue
                center_x = (other_left + other_right) / 2
                center_y = (other_top + other_bottom) / 2
                center_inside = left <= center_x <= right and top <= center_y <= bottom
                overlap_width = max(0, min(right, other_right) - max(left, other_left))
                overlap_height = max(0, min(bottom, other_bottom) - max(top, other_top))
                other_area = max(1, (other_right - other_left) * (other_bottom - other_top))
                ratio = overlap_width * overlap_height / other_area
                if center_inside or ratio >= threshold:
                    ambiguous = True
                    break

            # Them vien de anh vuong roi resize
            crop = image.crop((left, top, right, bottom))
            crop_width, crop_height = crop.size
            side = max(crop_width, crop_height)
            pad_left = (side - crop_width) // 2
            pad_top = (side - crop_height) // 2
            pad_right = side - crop_width - pad_left
            pad_bottom = side - crop_height - pad_top
            crop = ImageOps.expand(
                crop, (pad_left, pad_top, pad_right, pad_bottom), fill=(114, 114, 114)
            )
            crop = crop.resize((size, size), Image.Resampling.LANCZOS)

            group = "ambiguous" if ambiguous else "clean"
            output_name = f"{xml_path.stem}_obj{index}_{label}.jpg"
            output_path = Path(f"data/train/{country}/processed_raw/{group}/{label}") / output_name
            crop.save(output_path, quality=95, subsampling=0)
            counts[group, label] += 1
            rows.append([
                filename, xml_path.name, index, label,
                round(xmin), round(ymin), round(xmax), round(ymax),
                left, top, right, bottom, ambiguous, output_path.as_posix()
            ])

    # Luu thong tin cac anh da cat
    with open(f"data/train/{country}/processed_raw/manifest.csv", "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f)
        writer.writerow([
            "source_image", "source_xml", "object_index", "class_name",
            "xmin", "ymin", "xmax", "ymax", "crop_left", "crop_top",
            "crop_right", "crop_bottom", "ambiguous", "output_path"
        ])
        writer.writerows(rows)

    for label in classes:
        print(f"{country} {label}: clean={counts['clean', label]}, ambiguous={counts['ambiguous', label]}")
    print(f"Manifest: data/train/{country}/processed_raw/manifest.csv")

D00: clean=3150, ambiguous=899
D10: clean=2468, ambiguous=1511
D20: clean=3452, ambiguous=2746
D40: clean=1976, ambiguous=267
Manifest: data/train/Japan/processed_raw/manifest.csv


In [2]:
from pathlib import Path

remove = [
    "Japan_000073_obj1_D40.jpg",
    "Japan_000655_obj0_D40.jpg",
    "Japan_006823_obj0_D40.jpg",
    "Japan_008092_obj0_D40.jpg",
    "Japan_003187_obj0_D40.jpg",
    "Japan_010664_obj0_D20.jpg",
    "Japan_005541_obj2_D40.jpg",
    "Japan_009190_obj0_D00.jpg",
    "Japan_006089_obj0_D20.jpg",
    "Japan_011217_obj1_D20.jpg",
]

removed = 0
for filename in remove:
    label = filename.split("_")[-1].replace(".jpg", "")
    path = Path(f"data/train/Japan/processed_raw/clean/{label}/{filename}")
    if path.exists():
        path.unlink()
        removed += 1

print("Số ảnh đã xóa:", removed)

Số ảnh đã xóa: 10
